# Lezione 05 — Pipeline riproducibili

**Corso:** MLOps e Monitoraggio con Python · **Blocco 2:** Versionare e misurare
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/05_pipelines_reproducibles_IT.md`

## Cosa vedrai in questa lezione

- Costruire una `Pipeline` di scikit-learn **completa** (pulizia, `ColumnTransformer` per colonne numeriche/categoriche, modello) che si addestra e si applica con **un'unica chiamata**.
- Un caso reale di **fuga di informazione** (sovracampionare prima di dividere i dati): confrontiamo la metrica ottimistica con quella onesta, con numeri concreti.
- **Orchestrazione leggera**: quattro funzioni Python (`preparar`, `entrenar`, `evaluar`, `registrar`) chiamate da un unico script di ingresso, `main.py`, eseguito con `subprocess.run`.
- Cos'è un **feature store**, a livello di concetto, e perché questo corso non lo implementa.
- Il **training-serving skew** (la discrepanza tra addestramento e servizio), con un esempio piccolo e verificabile, e il suo legame con il feature store.

Tutto avviene dentro una cartella di lavoro effimera (`proyecto_churn/`), creata all'inizio ed **eliminata alla fine** di questo stesso notebook.

## Prerequisiti

- `pandas` e `scikit-learn` (la cella seguente installa solo ciò che manca, anche in Google Colab).
- La cartella `datos/` del corso, con `churn.csv` all'interno, e aver letto `datos/LEEME.md`.
- Dalla lezione 4: la configurazione 3 (regressione logistica, `class_weight="balanced"`, `C=1,0`), che riusiamo qui.

> **Come usare questo notebook:** esegui le celle in ordine, dall'alto in basso, una sola volta. Richiede meno di un minuto in totale.

---
## Installazione

Installa solo ciò che manca.

In [ ]:
import subprocess
import sys


def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("joblib", "joblib")
print("Librerie pronte.")

---
## Cella 0 — Setup

Importiamo le librerie, definiamo `cargar_datos` e fissiamo `SEMILLA = 42` per tutta la lezione.

In [ ]:
import os
import shutil
import subprocess
import sys
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

SEMILLA = 42
COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]

print("Librerie e funzioni ausiliarie pronte.")

---
## Cartella di lavoro effimera

Come nelle lezioni precedenti: creiamo `proyecto_churn/` da zero.

In [ ]:
CARPETA = "proyecto_churn"

# Puliamo e ricreiamo la cartella perché il notebook sia ripetibile da zero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Cartella di lavoro creata:", os.path.abspath(CARPETA))

---
## 1. Caricare i dati e dividere train/test

Carichiamo `churn.csv` e dividiamo PRIMA di costruire qualsiasi passo che impari dai dati.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y = df["abandono"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA, stratify=y
)

print("righe totali:", len(df))
print("righe di addestramento / test:", len(X_train), "/", len(X_test))
assert len(X_train) + len(X_test) == len(df)

---
## 2. La Pipeline completa: pulizia + ColumnTransformer + modello

Un'unica `Pipeline` incatena: imputazione (pulizia), `ColumnTransformer` (un ramo per le categoriche con `OneHotEncoder`, uno per le numeriche con `StandardScaler`) e il modello (regressione logistica bilanciata). `pipeline.fit(X_train, y_train)` addestra TUTTI i passi con un'unica chiamata.

In [ ]:
def construir_pipeline():
    """Pulizia (imputare) + ColumnTransformer (categoriche/numeriche) + modello, in un unico oggetto."""
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
    ])


# Un'unica chiamata addestra TUTTI i passi (imputatori, scaler, codificatore, modello)
pipeline = construir_pipeline()
pipeline.fit(X_train, y_train)

nombres_columnas = list(pipeline.named_steps["preprocesado"].get_feature_names_out())
print("colonne che il modello vede dopo il preprocessamento:", len(nombres_columnas))
print(nombres_columnas)
assert len(nombres_columnas) == 12

**Cosa osservi?** Il modello vede 12 colonne: 8 dell'`OneHotEncoder` (3 `plan` + 2 `contrato` + 3 `metodo_pago`) e 4 numeriche scalate — si può contare a mano in `COLUMNAS_CATEGORICAS`/`COLUMNAS_NUMERICAS` e verificarlo con l'`assert`.

In [ ]:
# Un'unica chiamata applica TUTTI i passi già adattati, nello stesso ordine
predicciones = pipeline.predict(X_test)
probabilidades = pipeline.predict_proba(X_test)[:, 1]

exactitud = round(accuracy_score(y_test, predicciones), 4)
f1 = round(f1_score(y_test, predicciones), 4)
auc = round(roc_auc_score(y_test, probabilidades), 4)

print("accuratezza:", exactitud)
print("f1:", f1)
print("auc:", auc)
assert (exactitud, f1, auc) == (0.6547, 0.3166, 0.7227), "dovrebbero coincidere con la configurazione 3 della lezione 4"

**Cosa osservi?** `pipeline.predict(X_test)` applica, con un'unica chiamata, TUTTI i passi già adattati (imputatori, scaler, codificatore, modello). Le cifre (accuratezza 0,6547, F1 0,3166, AUC 0,7227) coincidono con la configurazione 3 della lezione 4, perché è la stessa ricetta.

---
### 2.2 Perché la pulizia va sempre, anche se oggi non fa nulla

`churn.csv` non ha buchi, ma corrompiamo apposta una riga (un valore mancante reale) e confrontiamo la Pipeline con `SimpleImputer` rispetto a una versione senza.

In [ ]:
# Corrompiamo apposta una riga del set di test (un valore mancante reale)
X_con_hueco = X_test.copy()
X_con_hueco.iloc[0, X_con_hueco.columns.get_loc("uso_mensual_horas")] = np.nan

prediccion_con_imputer = pipeline.predict(X_con_hueco.iloc[[0]])
print("con SimpleImputer, predizione della riga con NaN:", prediccion_con_imputer)

# Pipeline SENZA imputatore, per confronto (stessa ricetta, senza il passo "imputar")
preprocesado_sin_imputar = ColumnTransformer([
    ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
    ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
])
pipeline_sin_imputar = Pipeline([
    ("preprocesado", preprocesado_sin_imputar),
    ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
])
pipeline_sin_imputar.fit(X_train, y_train)

try:
    pipeline_sin_imputar.predict(X_con_hueco.iloc[[0]])
    fallo = False
except ValueError as error:
    fallo = True
    print("senza SimpleImputer, ValueError:", str(error)[:40] + "...")

assert fallo, "senza imputatore, la Pipeline dovrebbe fallire con un valore mancante"

**Cosa osservi?** Con `SimpleImputer`, la riga con `NaN` si predice normalmente. Senza di esso, la stessa riga lancia `ValueError: Input X contains NaN` e ferma l'intero flusso. La pulizia non è un ornamento: decide se un solo valore mancante in produzione rompe una predizione o passa inosservato.

---
## 3. Un caso reale di fuga di informazione: sovracampionare prima di dividere

Hai già visto la fuga di informazione, con un altro meccanismo, nel corso di NLP classico (lezione 5). Qui usiamo un meccanismo diverso e altrettanto reale: duplicare la classe minoritaria (`abandono=1`) PRIMA di dividere in addestramento/test, per "bilanciare" il dataset. Il problema: copie quasi identiche di una riga di addestramento possono finire nel set di test.

In [ ]:
FACTOR_SOBREMUESTREO = 7


def sobremuestrear(df_en, semilla):
    """Duplica la classe minoritaria (abandono=1) con un po' di rumore, FACTOR_SOBREMUESTREO volte."""
    rng = np.random.RandomState(semilla)
    minoria = df_en[df_en["abandono"] == 1]
    copias = []
    for _ in range(FACTOR_SOBREMUESTREO):
        copia = minoria.copy()
        for columna in COLUMNAS_NUMERICAS:
            copia[columna] = copia[columna] * (1 + rng.normal(0, 0.01, size=len(copia)))
        copias.append(copia)
    return pd.concat([df_en] + copias, ignore_index=True)

In [ ]:
# APPROCCIO OTTIMISTICO (con fuga): sovracampioniamo TUTTO il dataset PRIMA di dividere
df_fugado = sobremuestrear(df, SEMILLA)
print("righe dopo aver sovracampionato l'intero dataset:", len(df_fugado))

X_fugado = df_fugado[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y_fugado = df_fugado["abandono"]
X_fug_train, X_fug_test, y_fug_train, y_fug_test = train_test_split(
    X_fugado, y_fugado, test_size=0.25, random_state=SEMILLA, stratify=y_fugado
)

pipeline_optimista = construir_pipeline()
pipeline_optimista.fit(X_fug_train, y_fug_train)
pred_optimista = pipeline_optimista.predict(X_fug_test)
proba_optimista = pipeline_optimista.predict_proba(X_fug_test)[:, 1]

exactitud_optimista = round(accuracy_score(y_fug_test, pred_optimista), 4)
f1_optimista = round(f1_score(y_fug_test, pred_optimista), 4)
auc_optimista = round(roc_auc_score(y_fug_test, proba_optimista), 4)

print("(ottimistico) righe train/test:", len(X_fug_train), "/", len(X_fug_test))
print("(ottimistico) accuratezza/f1/auc:", exactitud_optimista, f1_optimista, auc_optimista)

**Cosa osservi?** Dopo aver sovracampionato TUTTO il dataset, lo si divide in train/test — ma diverse copie quasi identiche della stessa riga originale possono finire una in train e un'altra in test. Il modello "riconosce" quelle righe invece di predirle davvero.

In [ ]:
# APPROCCIO ONESTO: dividiamo prima con i dati originali, sovracampioniamo SOLO l'addestramento
train_honesto = X_train.copy()
train_honesto["abandono"] = y_train.values
train_honesto_aumentado = sobremuestrear(train_honesto, SEMILLA)

X_hon_train = train_honesto_aumentado[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y_hon_train = train_honesto_aumentado["abandono"]

pipeline_honesto = construir_pipeline()
pipeline_honesto.fit(X_hon_train, y_hon_train)
# Il set di test (X_test, y_test) è quello ORIGINALE, intatto: non è mai passato per sobremuestrear()
pred_honesta = pipeline_honesto.predict(X_test)
proba_honesta = pipeline_honesto.predict_proba(X_test)[:, 1]

exactitud_honesta = round(accuracy_score(y_test, pred_honesta), 4)
f1_honesto = round(f1_score(y_test, pred_honesta), 4)
auc_honesto = round(roc_auc_score(y_test, proba_honesta), 4)

print("(onesto) righe train aumentato / test intatto:", len(X_hon_train), "/", len(X_test))
print("(onesto) accuratezza/f1/auc:", exactitud_honesta, f1_honesto, auc_honesto)

veces_inflado = round(f1_optimista / f1_honesto, 2)
print("l'F1 ottimistico è, volte quello onesto:", veces_inflado)
assert veces_inflado > 1.5, "la fuga dovrebbe gonfiare chiaramente l'F1"

**Cosa osservi?** L'F1 ottimistico (con fuga) è 2,14 volte l'F1 onesto: sembra il doppio di buono di quanto sia realmente. L'AUC cambia appena (0,7207 contro 0,7229): confronta l'ordine globale delle probabilità, cosa che le copie quasi identiche non alterano molto; l'F1, con una soglia fissa su predizioni concrete, nota invece in pieno il riconoscimento. Nessuna metrica da sola basta per fidarsi di un risultato.

---
## 4. Orchestrazione leggera: `main.py` con `preparar → entrenar → evaluar → registrar`

Scriviamo uno script reale, `proyecto_churn/main.py`, con quattro funzioni (una per fase) e una funzione `main()` che le chiama in ordine. Lo eseguiamo con `subprocess.run`, esattamente come si eseguirebbe da un terminale.

In [ ]:
MAIN_PY = """
import os
import sys
import json
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]
SEMILLA = 42
CARPETA = "proyecto_churn"


def cargar_datos(nombre):
    '''Cerca un file di dati in datos/, ../datos/ e /content/datos/.'''
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError("Non trovo " + nombre)


def preparar():
    '''Fase 1: carica churn.csv e separa addestramento/test.'''
    print("[preparar] caricamento di churn.csv e divisione train/test...")
    ruta = cargar_datos("churn.csv")
    df = pd.read_csv(ruta)
    X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
    y = df["abandono"]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=SEMILLA, stratify=y)
    print(f"[preparar] righe train={len(X_train)} test={len(X_test)}")
    return X_train, X_test, y_train, y_test


def construir_pipeline():
    '''Stessa ricetta del notebook: pulizia + ColumnTransformer + modello.'''
    preprocesado = ColumnTransformer([
        ("categoricas", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore")),
        ]), COLUMNAS_CATEGORICAS),
        ("numericas", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler()),
        ]), COLUMNAS_NUMERICAS),
    ])
    return Pipeline([
        ("preprocesado", preprocesado),
        ("modelo", LogisticRegression(class_weight="balanced", random_state=SEMILLA, max_iter=1000)),
    ])


def entrenar(X_train, y_train):
    '''Fase 2: adatta la Pipeline completa con un'unica chiamata.'''
    print("[entrenar] adattamento della Pipeline completa (preprocessamento + modello)...")
    pipeline = construir_pipeline()
    pipeline.fit(X_train, y_train)
    print("[entrenar] pipeline.fit() completato.")
    return pipeline


def evaluar(pipeline, X_test, y_test):
    '''Fase 3: applica la Pipeline al set di test con un'unica chiamata.'''
    print("[evaluar] calcolo delle metriche sul set di test...")
    predicciones = pipeline.predict(X_test)
    probabilidades = pipeline.predict_proba(X_test)[:, 1]
    metricas = {
        "exactitud": round(accuracy_score(y_test, predicciones), 4),
        "f1": round(f1_score(y_test, predicciones), 4),
        "auc": round(roc_auc_score(y_test, probabilidades), 4),
    }
    print("[evaluar] metriche:", metricas)
    return metricas


def registrar(pipeline, metricas):
    '''Fase 4: salva il modello addestrato e le sue metriche nella cartella effimera.'''
    print("[registrar] salvataggio del modello e dei suoi metadati...")
    ruta_modelo = os.path.join(CARPETA, "modelo.joblib")
    joblib.dump(pipeline, ruta_modelo)
    ruta_metricas = os.path.join(CARPETA, "metricas.json")
    with open(ruta_metricas, "w") as f:
        json.dump(metricas, f, indent=2)
    print(f"[registrar] modello salvato in {ruta_modelo}")
    print(f"[registrar] metriche salvate in {ruta_metricas}")


def main():
    X_train, X_test, y_train, y_test = preparar()
    pipeline = entrenar(X_train, y_train)
    metricas = evaluar(pipeline, X_test, y_test)
    registrar(pipeline, metricas)
    print("[main] orchestrazione completa: preparar -> entrenar -> evaluar -> registrar.")


if __name__ == "__main__":
    main()
"""

In [ ]:
ruta_main = os.path.join(CARPETA, "main.py")
with open(ruta_main, "w", encoding="utf-8") as f:
    f.write(MAIN_PY)
print("script di ingresso creato in:", ruta_main)

In [ ]:
# Eseguiamo lo script come si farebbe da un terminale o da un flusso di CI/CD
resultado = subprocess.run([sys.executable, ruta_main], capture_output=True, text=True)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
assert resultado.returncode == 0, "lo script main.py dovrebbe terminare senza errori"

**Cosa osservi?** L'output mostra le quattro fasi in ordine (`[preparar]`, `[entrenar]`, `[evaluar]`, `[registrar]`), esattamente come le abbiamo scritte in `main.py`. Lo script termina con codice di uscita 0 (senza errori), come verificherebbe una CI/CD reale.

In [ ]:
with open(os.path.join(CARPETA, "metricas.json")) as f:
    metricas_script = json.load(f)
print("metriche lette dal file registrato da main.py:", metricas_script)

assert metricas_script == {"exactitud": exactitud, "f1": f1, "auc": auc}, "dovrebbero coincidere con quelle calcolate a mano nella sezione 2"
assert os.path.exists(os.path.join(CARPETA, "modelo.joblib"))
print("coincidono con quelle della sezione 2: è la stessa Pipeline, eseguita due volte con lo stesso seme.")

---
## 5. Feature store: concetto, e perché qui non lo implementiamo

Un **feature store** calcola, salva e serve le variabili di un modello una sola volta, in modo che addestramento e produzione leggano sempre lo stesso valore. Non lo implementiamo in questo corso: richiede infrastruttura propria (database condiviso, servizio in tempo reale) che non rientra in un ambiente locale/Colab con un solo progetto. Il capitolo del manuale (sezione 5) ha la tabella completa di confronto.

---
## 6. Training-serving skew: un esempio piccolo e verificabile

Simuliamo due reimplementazioni della stessa variabile (`antiguedad_meses` raggruppata in categorie): una nel codice di "addestramento" e un'altra, con un errore di limite (`<` invece di `<=`), nel codice di "servizio".

In [ ]:
def categoria_entrenamiento(meses):
    """Codice di addestramento: 'medio' include il limite superiore (<=36)."""
    if meses < 12:
        return "nuevo"
    elif meses <= 36:
        return "medio"
    return "antiguo"


def categoria_servicio(meses):
    """Reimplementazione nel servizio, con un errore di limite (<36 invece di <=36)."""
    if meses < 12:
        return "nuevo"
    elif meses < 36:
        return "medio"
    return "antiguo"


categoria_train = df["antiguedad_meses"].apply(categoria_entrenamiento)
categoria_serv = df["antiguedad_meses"].apply(categoria_servicio)
filas_distintas = int((categoria_train != categoria_serv).sum())
porcentaje = round(filas_distintas / len(df) * 100, 2)

print("righe con categoria diversa tra addestramento e servizio:", filas_distintas, "/", len(df), f"({porcentaje}%)")
assert filas_distintas == 50

**Cosa osservi?** Solo 50 righe su 3 000 (1,67 %) hanno `antiguedad_meses == 36` esattamente, e sono le uniche dove le due funzioni differiscono. Nessun errore, nessuna eccezione: un training-serving skew silenzioso che si rileva solo confrontando entrambe le implementazioni riga per riga. Un feature store elimina questo rischio perché la variabile si calcola in un unico posto.

---
## 7. Pulizia finale

Eliminiamo `proyecto_churn/` (con `main.py`, il modello e le metriche) perché non resti nessun residuo dentro il repository del corso.

In [ ]:
shutil.rmtree(CARPETA)
print("Cartella di lavoro eliminata:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Riepilogo della lezione

- Una `Pipeline` di scikit-learn completa (pulizia + `ColumnTransformer` + modello) si addestra e si applica con un'unica chiamata ogni volta, senza ripetere passi a mano.
- La fuga di informazione compare anche sovracampionando prima di dividere: l'F1 "ottimistico" è stato **2,14 volte** quello onesto, anche se l'AUC è cambiato appena — lo stesso tipo di problema visto in NLP lezione 5, con un altro meccanismo.
- L'**orchestrazione leggera** (`preparar → entrenar → evaluar → registrar`, in `main.py`) copre l'essenziale di un flusso di lavoro senza bisogno di un orchestratore pesante.
- Un **feature store** evita che la stessa variabile si reimplementi in più punti; senza di esso, il **training-serving skew** può comparire con un semplice errore di limite, come mostra l'esempio dell'1,67 % di righe colpite.

*Continua con la lezione 6: Testare i sistemi di ML.*